# Activity 8.2 — Detect and Mask PII Using NLP

*Chapter 8 — AI Security and Vulnerability Testing · 40 minutes · individual, then pairs*

The support team got the green light after Activity 8.1: the structured columns were
minimized — but the `body` field stays, and free text carries whatever the customer
typed. Before tickets may go to the external drafting model, the PII **inside the body**
must be found and masked. You will build two detectors — pattern matching and a semantic
(LLM) pass — compare what each catches, then drive a redactor with the union of both.

## Objectives

By the end of this activity, you will:

- Catch shaped PII (emails, phone numbers, order numbers) in free text with
  pattern matching.
- Catch contextual PII (names, health details, addresses) with an LLM semantic
  pass that returns structured output.
- Explain, using the clean-ticket control, why each pass sees what the other misses.
- Drive a redactor from the union of both passes and prove the masked output clean.


## Setup — the tickets

Same `support_tickets_raw.csv` as Activity 8.1: twelve tickets from the support queue.
Here we scan only the `body` column — Activity 8.1 already dealt with the structured
fields.

In [ ]:
import csv
import re
import textwrap

import course_ai

print("mode:", course_ai.mode())   # 'mock' (offline) or 'live' (OPENAI_API_KEY set)
SHOW = lambda t: print(textwrap.fill(str(t), 100))

with open("support_tickets_raw.csv", newline="", encoding="utf-8") as f:
    tickets = list(csv.DictReader(f))

print(f"{len(tickets)} tickets loaded — sample body:")
SHOW(tickets[0]["body"])

## Part A — Pattern matching (10 min)

Some PII has a fixed shape, and a regular expression finds it fast and for free:
email addresses, North-American phone numbers, our `LT-#####` order numbers, and the
phrase "card ending NNNN". Run the sweep over every ticket body.

In [ ]:
PATTERNS = {
    "email":        re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "phone":        re.compile(r"\b\d{3}-\d{3}-\d{4}\b"),
    "order_number": re.compile(r"\bLT-\d{5}\b"),
    "card_last4":   re.compile(r"ending\s+(\d{4})"),
}


def regex_scan(text):
    """Return {'type': [values]} for every pattern that matches `text`."""
    hits = {}
    for kind, pat in PATTERNS.items():
        vals = [m.group(1) if pat.groups else m.group(0) for m in pat.finditer(text)]
        if vals:
            hits[kind] = vals
    return hits


for t in tickets:
    hits = regex_scan(t["body"])
    print(f"{t['ticket_id']}: {hits if hits else '—'}")

**Checkpoint** — pattern matching flagged the *shaped* PII. Now read the bodies of
T-1005 through T-1008 and T-1010 with your own eyes: what did the regex **not** see?
A name in running text, a health note, a street address — none of these has a shape you
can write down. That blind spot is what the semantic pass closes.

## Part B — Semantic detection with an LLM (15 min)

No regex catches *"My manager Greg will follow up"* — but a model that reads the text
can. Ask it for structured output: one JSON list of PII entities per ticket. (In mock
mode the reply is a fixed transcript of what a competent model returns for these twelve
tickets — including the contextual PII above.)

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "value": {"type": "string"},
                },
                "required": ["type", "value"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["entities"],
    "additionalProperties": False,
}

PROMPT = """You are a PII detection engine. List every piece of personally
identifiable information in TICKET {tid} below: person names, email addresses,
phone numbers, postal addresses, order or account numbers, payment details,
and health or other sensitive personal information. Return an empty list if
there is none.

TICKET {tid}:
{body}"""


def llm_scan(ticket):
    out = course_ai.chat_json(
        PROMPT.format(tid=ticket["ticket_id"], body=ticket["body"]), SCHEMA)
    return out["entities"]


for t in tickets:
    print(f"{t['ticket_id']}: {llm_scan(t) or '—'}")

## Side by side — what each pass sees

Line the two sweeps up. Pay attention to the tickets where the LLM finds something and
the regex found nothing at all — and to **T-1009**, the control ticket that both passes
(correctly) leave alone.

In [ ]:
print(f"{'ticket':8} {'regex found':28} llm found")
print("-" * 78)
contextual = []
for t in tickets:
    rx = {(k, v) for k, vals in regex_scan(t["body"]).items() for v in vals}
    ll = {(e["type"], e["value"]) for e in llm_scan(t)}
    if ll and not rx:
        contextual.append(t["ticket_id"])
    fmt = lambda s: ", ".join(sorted({k for k, _ in s})) or "—"
    print(f"{t['ticket_id']:8} {fmt(rx):28} {fmt(ll)}")

# The lesson, checked: several tickets carry PII ONLY the semantic pass can see.
assert len(contextual) >= 3, f"expected >=3 regex-blind tickets, got {contextual}"
print(f"\nregex-blind tickets (LLM-only PII): {', '.join(contextual)}")

## Part C — Mask and re-scan (10 min)

A detector that cannot drive a redactor is half a control. Take the **union** of both
passes, replace every entity with a typed placeholder, then prove the sweep is clean by
re-running the regex pass over the masked text. *This* output is what may leave the
building.

In [ ]:
def entities_for(ticket):
    ents = {(k, v) for k, vals in regex_scan(ticket["body"]).items() for v in vals}
    ents |= {(e["type"], e["value"]) for e in llm_scan(ticket)}
    return [{"type": k, "value": v} for k, v in sorted(ents)]


def mask(text, entities):
    for e in sorted(entities, key=lambda e: -len(e["value"])):  # longest first
        text = text.replace(e["value"], f"[{e['type'].upper()}]")
    return text


masked = {t["ticket_id"]: mask(t["body"], entities_for(t)) for t in tickets}

# Proof: no pattern-shaped PII survives in any masked body.
leftovers = {tid: regex_scan(b) for tid, b in masked.items()}
assert not any(leftovers.values()), f"PII survived masking: {leftovers}"
print("re-scan clean: no regex-detectable PII in any masked body\n")

before = next(t for t in tickets if t["ticket_id"] == "T-1008")
SHOW("BEFORE: " + before["body"])
SHOW("AFTER:  " + masked["T-1008"])

## Reflection (5 minutes, pairs)

- Which entities did only the LLM catch — and why can no regex ever catch them?
- The masked tickets are **pseudonymous, not anonymous**: combine the remaining fields
  (product, version, issue, dates) with outside knowledge and some customers could still
  be re-identified. What else would you strip before calling this dataset anonymous?
- T-1009 came back clean on both passes. Why does a verified-negative control matter for
  a control like this?
- Where in *your* pipeline would this sweep run: before the prompt is built, or on the
  vendor's response before it is shown to the customer?

**Takeaway:** pattern matching catches the obvious PII; the interesting failures are
contextual — a name in free text, a health note in a billing ticket. Detection (both
passes) plus masking is the free-text half of data minimization you started in 8.1.